# Stage 23D: hierarchical emission decoder

Stage 23BのrankerとStage 23Cで保存したOOF posteriorを再利用し、offsetを直接回帰せず、移動有無・方向・絶対量に分けて校正します。Stage 23Cのvolumeを再計算せず、GPUも不要です。Stage 21BはすでにStage 23B/Cの設計判断に使われているため、ここではdesign validationとして扱います。通過してもKaggle提出せず、新しいwell集合で確認します。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json,os,shutil,subprocess
REPOSITORY_URL='https://github.com/Okada-N13/rogii.git'
repo_dir=Path('/content/ROGII'); drive_root=Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir=drive_root/'artifacts'
if not (repo_dir/'.git').is_dir(): subprocess.run(['git','clone',REPOSITORY_URL,str(repo_dir)],check=True)
else: subprocess.run(['git','-C',str(repo_dir),'pull','--ff-only','origin','main'],check=True)
if shutil.which('uv') is None: subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],check=True)
os.environ['PATH']='/root/.local/bin:'+os.environ['PATH']
subprocess.run(['uv','sync','--frozen'],cwd=repo_dir,check=True)
print('Stage 23D uses CPU and reuses Stage 23C parquet artifacts')


## Stage 23C artifacts

training OOF 77 cutsとdesign validation 62 cutsのposterior特徴をそのまま使います。


In [ ]:
stage23c_run=artifact_dir/'stage23c_oof_decoder_full_v001'
required=[stage23c_run/'summary.json',stage23c_run/'training_oof_decoder_rows.parquet',stage23c_run/'validation_decoder_rows.parquet']
for path in required: assert path.is_file(),path
stage23c_summary=json.loads((stage23c_run/'summary.json').read_text())
assert stage23c_summary['stage23c_complete'] and not stage23c_summary['promoted_to_stage23d']
print(*required,sep='\n')


## Nested selection and design validation

4つの事前固定profileをtraining OOF内でnested比較します。eligible profileがなければvalidation予測はbaseと同一になり、自動的に不合格です。


In [ ]:
RUN_ID='stage23d_hierarchical_decoder_full_v001'; run_dir=artifact_dir/RUN_ID
if run_dir.exists() and not (run_dir/'summary.json').is_file():
    resolved=run_dir.resolve(); expected=(artifact_dir/RUN_ID).resolve()
    assert resolved==expected and resolved.parent==artifact_dir.resolve(),resolved
    print('Removing incomplete prior run:',resolved); shutil.rmtree(resolved)
if not (run_dir/'summary.json').is_file():
    command=['uv','run','rogii-emission-hierarchical-decoder','--config','configs/experiment/stage23d_hierarchical_decoder.yaml','--stage23c-run',str(stage23c_run),'--artifact-dir',str(artifact_dir),'--run-id',RUN_ID]
    result=subprocess.run(command,cwd=repo_dir,text=True,capture_output=True)
    if result.stdout: print(result.stdout)
    if result.returncode:
        print(result.stderr); raise RuntimeError(f'Stage 23D failed: {command}')
summary=json.loads((run_dir/'summary.json').read_text())
{key:summary[key] for key in ['stage23d_complete','promoted_to_stage23e_disjoint_confirmation','training_cuts','training_wells','validation_cuts','validation_wells','training_validation_well_overlap','selected_profile','validation_base_rmse','validation_candidate_rmse','validation_delta','validation_p90_delta','validation_bootstrap_95pct','validation_role','gates','next_step']}


In [ ]:
import pandas as pd
pd.DataFrame(summary['nested_profile_report']).sort_values(['eligible','rmse'],ascending=[False,True])


最後のsummary辞書とnested profile表を共有してください。通過しても、この結果からprofileを差し替えたりKaggle提出を作ったりしません。
